# Lab 2.1 - Find Similar Documents with Embeddings (Start)
**Day 2 - Session 1**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/day2/lab2_1_similarity/start/lab2_1_similarity.ipynb)

**Goal:** Embed a small set of documents with a sentence-transformers model,
then rank them by cosine similarity to a query. Compare semantic search with
plain keyword matching.

Look for the `# TODO` markers and fill them in. The notebook in the `solution/`
folder has the full answer if you get stuck.

Everything runs on a Colab CPU runtime in under a minute; no API key and no
GPU needed.

In [ ]:
# Scoped install for this lab (safe to re-run; preinstalled on Colab).
%pip install -q sentence-transformers scikit-learn

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

np.random.seed(42)

## 1. A small document collection
Eight short help-desk style documents. Small enough to read, varied enough to
show that meaning beats keyword overlap.

In [ ]:
DOCUMENTS = [
    "How to reset your account password from the login screen.",
    "Steps to update your billing information and payment method.",
    "Troubleshooting slow application startup on older laptops.",
    "How to invite a teammate and manage their permissions.",
    "Recovering access when you are locked out of your account.",
    "Exporting your monthly invoices as a PDF file.",
    "Fixing sync errors between the mobile app and the web app.",
    "Changing your notification settings and email preferences.",
]
for i, doc in enumerate(DOCUMENTS):
    print(f"{i}: {doc}")

## 2. Load an embedding model
`all-MiniLM-L6-v2` is small (about 80 MB) and fast on CPU. It maps each text
to a 384-dimension dense vector.

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

# TODO: Embed the documents. Pass normalize_embeddings=True so cosine similarity
# becomes a simple dot product. Assign the result to doc_embeddings.
# Hint: doc_embeddings = model.encode(DOCUMENTS, normalize_embeddings=True)
doc_embeddings = ...

print("Embeddings shape:", doc_embeddings.shape)

## 3. Rank documents by cosine similarity
Because the vectors are normalized, cosine similarity is a dot product. Higher
score means closer in meaning.

In [ ]:
def search(query, top_k=3):
    # TODO: Embed the query with normalize_embeddings=True and take row [0].
    query_vec = ...
    # TODO: Score every document against the query with a dot product.
    # Hint: scores = doc_embeddings @ query_vec
    scores = ...
    ranked = np.argsort(scores)[::-1]
    print(f"\nQuery: {query!r}")
    for rank, idx in enumerate(ranked[:top_k], start=1):
        print(f"  {rank}. score={scores[idx]:.3f}  {DOCUMENTS[idx]}")
    return ranked, scores


_ = search("I forgot my password and cannot log in")

## 4. Semantic search vs keyword matching
The query below shares no keywords with the best document. Keyword matching
misses it; embeddings still find it because the meaning is close.

In [ ]:
def keyword_search(query, top_k=3):
    query_words = set(query.lower().split())
    # TODO: For each document, count how many query words also appear in it.
    # Hint: overlap = [len(query_words & set(doc.lower().split())) for doc in DOCUMENTS]
    overlap = ...
    ranked = np.argsort(overlap)[::-1]
    print(f"\nKeyword query: {query!r}")
    for rank, idx in enumerate(ranked[:top_k], start=1):
        print(f"  {rank}. overlap={overlap[idx]}  {DOCUMENTS[idx]}")
    return ranked


QUERY = "locked out and need to regain entry"
print("=== Semantic search ===")
search(QUERY)
print("\n=== Keyword search ===")
keyword_search(QUERY)

Semantic search should rank "Recovering access when you are locked out of your
account" near the top. Keyword search struggles because the wording differs.

## Stretch goals
Try these if you finish early. Hints only here; the full code is in the
solution notebook.

1. **Cluster the documents.** Use `sklearn.cluster.KMeans` with 3 clusters on
   `doc_embeddings` and print the documents in each cluster.
2. **Compare a second model.** Load `all-MiniLM-L12-v2`, re-embed the
   documents, and check whether the top match for a query changes.

In [ ]:
# TODO (stretch): your experiments here.